# Example: MUSICA CO single-tracer transport

This notebook uses the routines in `quacs.tagged_tracers.voc_nox`.
The notebook only sets the inputs and plots the results.

The case:

1. `CO` has emissions in every grid cell.
2. `CO_tracer` has emissions only in one tagged source grid cell.
3. MUSICA does the local emissions and losses.
4. An explicit 1-D transport operator moves `CO` and `CO_tracer`.
5. The notebook checks a receptor grid cell for the source-to-receptor signal.

This case does not use plume rise, fire emissions, biogenic emissions, or MPAS transport.

In [ ]:
import matplotlib.pyplot as plt
import musica
import numpy as np
import pandas as pd
from IPython.display import display

from quacs.tagged_tracers.voc_nox import (
    build_1d_grid,
    build_co_tracer_mechanism,
    build_musica_rate_parameters,
    build_single_cell_tracer_emissions,
    build_total_co_emissions,
    derive_simple_met,
    history_to_dataframe,
    run_source_receptor,
    source_receptor_proof_table,
)

np.set_printoptions(suppress=True)
pd.set_option("display.float_format", str)

## 1. Define grid and tagged source/receptor cells

In [ ]:
num_grid_cells = 30

source_cell = 6
receptor_cell = 20

grid = build_1d_grid(
    num_grid_cells=num_grid_cells,
    source_cell=source_cell,
    receptor_cell=receptor_cell,
)

print(f"Number of MUSICA grid cells: {num_grid_cells}")
print(f"Tagged source cell: {source_cell}")
print(f"Receptor cell: {receptor_cell}")

## 2. Define meteorological inputs

In [ ]:
met = derive_simple_met(
    num_grid_cells=num_grid_cells,
    temperature=298.0,
    pressure=101325.0,
)

temperature = met["temperature"]
pressure = met["pressure"]

print("temperature shape:", temperature.shape)
print("pressure shape:", pressure.shape)

## 3. Build MUSICA mechanism

In [ ]:
mechanism = build_co_tracer_mechanism()

for reaction in mechanism.reactions:
    print(f"[{reaction.type.name}] {reaction.to_equation()}")

## 4. Create solver and state

In [ ]:
solver = musica.MICM(
    mechanism=mechanism,
    solver_type=musica.SolverType.rosenbrock_standard_order,
)

state = solver.create_state(num_grid_cells)

state.set_conditions(
    temperatures=temperature,
    pressures=pressure,
)

state.set_concentrations(
    {
        "CO": np.zeros(num_grid_cells),
        "CO_tracer": np.zeros(num_grid_cells),
    }
)

## 5. Build emissions and MUSICA rate parameters

In [ ]:
CO_emis = build_total_co_emissions(
    grid=grid,
    base_emis=1.0e-12,
    enhancement=5.0e-13,
    enhancement_center=12.0,
    enhancement_width=30.0,
)

CO_tracer_emis = build_single_cell_tracer_emissions(
    total_co_emis=CO_emis,
    source_cell=source_cell,
)

rate_parameters = build_musica_rate_parameters(
    co_emis=CO_emis,
    co_tracer_emis=CO_tracer_emis,
)

print("CO emission at receptor:", CO_emis[receptor_cell])
print("CO_tracer emission at source:", CO_tracer_emis[source_cell])
print("CO_tracer emission at receptor:", CO_tracer_emis[receptor_cell])

assert CO_tracer_emis[source_cell] > 0.0
assert CO_tracer_emis[receptor_cell] == 0.0

## 6. Run MUSICA + explicit transport

In [ ]:
dt = 10.0
sim_length = 3000.0
dx = 1.0

K_diff = 0.002
u_adv = 0.006

times = np.arange(0, sim_length + dt, dt)

CO_hist, CO_tracer_hist = run_source_receptor(
    solver,
    state,
    rate_parameters,
    times,
    dt=dt,
    K_diff=K_diff,
    u_adv=u_adv,
    dx=dx,
)

print("CO_hist shape:", CO_hist.shape)
print("CO_tracer_hist shape:", CO_tracer_hist.shape)

## 7. Source-receptor proof

In [ ]:
proof = source_receptor_proof_table(
    times=times,
    co_hist=CO_hist,
    co_tracer_hist=CO_tracer_hist,
    co_tracer_emis=CO_tracer_emis,
    source_cell=source_cell,
    receptor_cell=receptor_cell,
)

display(proof.head())
display(proof.tail())

print("Max tracer emission at receptor:", CO_tracer_emis[receptor_cell])
print("Max tracer concentration at receptor:", proof["CO_tracer_at_receptor"].max())

assert CO_tracer_emis[receptor_cell] == 0.0
assert proof["CO_tracer_at_receptor"].max() > 0.0

print("PASSED: receptor CO_tracer becomes nonzero even though receptor CO_tracer emission is zero.")

## 8. Spatial tracer evolution

In [ ]:
plt.figure(figsize=(10, 5))

time_indices = [
    0,
    int(0.20 * (len(times) - 1)),
    int(0.40 * (len(times) - 1)),
    int(0.60 * (len(times) - 1)),
    int(0.80 * (len(times) - 1)),
    len(times) - 1,
]

for ti in time_indices:
    plt.plot(
        grid,
        CO_tracer_hist[ti, :],
        marker="o",
        label=f"t = {times[ti]:.0f} s",
    )

plt.axvline(source_cell, linestyle="--", label="source cell")
plt.axvline(receptor_cell, linestyle=":", label="receptor cell")

plt.xlabel("Grid cell")
plt.ylabel("CO_tracer concentration (mol m$^{-3}$)")
plt.title("Tagged CO transport from source cell to receptor cell")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 9. Receptor time series

In [ ]:
plt.figure(figsize=(9, 4))

plt.plot(
    times,
    CO_tracer_hist[:, source_cell],
    label=f"CO_tracer at source cell {source_cell}",
)

plt.plot(
    times,
    CO_tracer_hist[:, receptor_cell],
    label=f"CO_tracer at receptor cell {receptor_cell}",
)

plt.xlabel("Time (s)")
plt.ylabel("CO_tracer concentration (mol m$^{-3}$)")
plt.title("Tagged CO source-to-receptor signal")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## 10. Heatmap

In [ ]:
plt.figure(figsize=(10, 5))

plt.imshow(
    CO_tracer_hist,
    aspect="auto",
    origin="lower",
    extent=[grid.min(), grid.max(), times.min(), times.max()],
)

plt.axvline(source_cell, linestyle="--", linewidth=2, label="source cell")
plt.axvline(receptor_cell, linestyle=":", linewidth=2, label="receptor cell")

plt.colorbar(label="CO_tracer concentration (mol m$^{-3}$)")
plt.xlabel("Grid cell")
plt.ylabel("Time (s)")
plt.title("CO_tracer transport from source to receptor")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Total CO vs tagged source contribution at receptor

In [ ]:
plt.figure(figsize=(9, 4))

plt.plot(
    times,
    CO_hist[:, receptor_cell],
    label=f"Total CO at receptor cell {receptor_cell}",
)

plt.plot(
    times,
    CO_tracer_hist[:, receptor_cell],
    label=f"Tagged source contribution from cell {source_cell}",
)

plt.xlabel("Time (s)")
plt.ylabel("Concentration (mol m$^{-3}$)")
plt.title("Total CO vs tagged source contribution at receptor")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

ratio = np.divide(
    CO_tracer_hist[:, receptor_cell],
    CO_hist[:, receptor_cell],
    out=np.zeros_like(times, dtype=float),
    where=CO_hist[:, receptor_cell] > 0,
)

plt.figure(figsize=(9, 4))
plt.plot(times, ratio)
plt.xlabel("Time (s)")
plt.ylabel("Tagged / total CO at receptor")
plt.title("Fraction of receptor CO from tagged source cell")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 12. Save outputs

Set `SAVE_OUTPUTS = True` to write the CSV files and the mechanism JSON file to the current folder.

In [ ]:
SAVE_OUTPUTS = False

df = history_to_dataframe(
    times=times,
    grid=grid,
    co_hist=CO_hist,
    co_tracer_hist=CO_tracer_hist,
    co_emis=CO_emis,
    co_tracer_emis=CO_tracer_emis,
    temperature=temperature,
    pressure=pressure,
    source_cell=source_cell,
    receptor_cell=receptor_cell,
)

display(df.head())

if SAVE_OUTPUTS:
    df.to_csv("ex_tagged_co_musica_transport_output.csv", index=False)
    proof.to_csv("ex_tagged_co_musica_transport_proof.csv", index=False)
    mechanism.export("ex_tagged_co_musica_transport_mechanism.json")
    print("Wrote the output files.")